In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar el dataset

In [2]:
penguins = sns.load_dataset("penguins").dropna()

# Seleccionar características y variable objetivo

In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
target = "body_mass_g"
X = penguins[features].values
y = penguins[target].values.reshape(-1, 1)

# Normalizamos los datos

In [8]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# Convertimos las variables a tensores de Pytorch

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
X_train, X_test = torch.tensor(X_train, dtype=torch.float32), torch.tensor(X_test, dtype=torch.float32)

In [11]:
y_train, y_test = torch.tensor(y_train, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32)

# Creamos modelo de red neuronal con Pytorch

In [12]:
class Net(nn.Module):
  def __init__(self):
    super(Net,self).__init__()
    self.fc1 = nn.Linear(3,16)
    self.fc2 = nn.Linear(16,8)
    self.fc3 = nn.Linear(8,1)
    self.relu = nn.ReLU()

  def forward(self,x):
    x = self.relu(self.fc1(x))
    x = self.relu(self.fc2(x))
    x = self.fc3(x)
    return x

## Configuramos modelo función de perdida y optimizador

In [13]:
model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

# Entrenamos nuestro modelo

In [14]:
epochs = 500
for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()
  outputs = model(X_train)
  loss = criterion(outputs,y_train)
  loss.backward()
  optimizer.step()

  if(epoch + 1) % 50 == 0:
    print(f'epoch [{epoch+1}/{epochs}], Loss :{loss.item():.4f}')

epoch [50/500], Loss :0.2066
epoch [100/500], Loss :0.1687
epoch [150/500], Loss :0.1568
epoch [200/500], Loss :0.1507
epoch [250/500], Loss :0.1441
epoch [300/500], Loss :0.1387
epoch [350/500], Loss :0.1344
epoch [400/500], Loss :0.1316
epoch [450/500], Loss :0.1283
epoch [500/500], Loss :0.1259


# Evaluamos el modelo

In [15]:
model.eval()
y_pred = model(X_test).detach().numpy()
y_pred = scaler_y.inverse_transform(y_pred)
y_test = scaler_y.inverse_transform(y_test.numpy())

for i in range(5):
  print(f'valor predicho {y_pred[i][0]:-2f} - Valor Real : {y_test[i][0]:.2f}')


valor predicho -1.153464 - Valor Real : -1.19
valor predicho 1.071995 - Valor Real : 0.83
valor predicho -0.129780 - Valor Real : -0.26
valor predicho -0.773395 - Valor Real : -0.66
valor predicho -0.451395 - Valor Real : -0.20


# Evaluar con métricas de sklearn

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 Score: {r2:.2f}")

MAE: 0.33
MSE: 0.19
R2 Score: 0.80
